In [4]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
with sqlite3.connect('../db.sqlite3') as conn:
    query = (
        "SELECT oi.*, o.datetime as order_datetime, "
        "p.name as product_name, p.price as product_price "
        "FROM restaurant_orderitem oi "
        "JOIN restaurant_order o ON oi.order_id = o.id "
        "JOIN restaurant_product p ON oi.product_id = p.id"
    )
    df = pd.read_sql_query(query, conn)
df.to_csv('../restaurant_data.csv', index=False)
df = pd.read_csv('../restaurant_data.csv')
print(f"Rows: {len(df)}")

Rows: 74818


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top10_quantity = (
    df.groupby('product_name')['quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
plt.figure(figsize=(10, 8))
plt.pie(
    top10_quantity.values,
    labels=top10_quantity.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Top 10 Most Popular Products by Quantity')
plt.axis('equal')
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df['item_price'] = df['quantity'] * df['product_price']
top10_item_price = df.groupby('product_name')['item_price'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 8))
plt.pie(top10_item_price.values, labels=top10_item_price.index, autopct='%1.1f%%', startangle=90)
plt.title('Top 10 Most Popular Products by Item Price')
plt.axis('equal')
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['order_hour'] = pd.to_datetime(df['order_datetime']).dt.hour
income_by_hour = df.groupby('order_hour')['item_price'].sum()
plt.figure(figsize=(12, 8))
plt.bar(income_by_hour.index, income_by_hour.values)
plt.xlabel('Order Hour')
plt.ylabel('Total Income')
plt.title('Total Restaurant Income by Order Hour')
plt.xticks(range(24))
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
# write your code here
df['order_day_of_week'] = pd.to_datetime(df['order_datetime']).dt.day_name()
income_by_day = df.groupby('order_day_of_week')['item_price'].sum()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
income_by_day = income_by_day.reindex(days_order)

plt.figure(figsize=(12, 8))
plt.bar(income_by_day.index, income_by_day.values)
plt.xlabel('Order day of week')
plt.ylabel('Total Income')
plt.title('Total Restaurant Income by Order Day of Week')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()